# DSv4 Pallas 커널 완전정복 — 0부터 천천히, **한 단위씩**

> 이 노트북은 [`kernel_v1.py`](./kernel_v1.py)(FlashAttention-with-sink 순전파/역전파)와
> [`kernel_v2.py`](./kernel_v2.py)(전체 CSA/HCA 조립 + Sinkhorn 커널)를 **하나의 실행 가능한
> 튜토리얼**로 묶은 것입니다. 커널을 한 번도 안 짜본 사람도 위에서부터 차례로 읽고 셀을
> 실행하면, 마지막 Sinkhorn 역전파까지 도달합니다.

**읽는 법** — 난이도 레벨이 아니라 *개념 단위*로 쭉 이어집니다. 각 개념은 **그게 실제로
필요해지는 자리에서, 한 번에 끝까지** 설명됩니다.

- **A. 왜 이 커널이 필요한가** — 평범한 attention의 메모리 통증 + TPU 한 장의 구조
- **B. Pallas 도구** — 타일·grid·BlockSpec, 그리고 첫 호출, 블랙박스로 한 번 돌려보기
- **C. 무엇을 계산하나** — sparse MQA + V==K + attention sink, 식 하나로
- **D. V1 순전파** — gather 껍데기 → grid 스트리밍 → ★ **online softmax 완전 설명** → 커널 본체 → `_dot2`
- **E. V1 역전파** — 저장한 `lse`로 `p` 복원 + softmax 역전파 항등식 + custom_vjp
- **F. V2 조립** — 전체 CSA/HCA forward 파이프라인
- **G. V2 Sinkhorn** — doubly stochastic부터 Pallas 안의 `jax.vjp`까지
- **부록** — Sinkhorn 수렴 직관 / 자주 헷갈리는 점 / 더 읽을거리

대상 파일: [`kernel_v1.py`](./kernel_v1.py), [`kernel_v2.py`](./kernel_v2.py),
[`kernel.py`](./kernel.py), [`eager.py`](./eager.py), [`kernel_config.py`](./kernel_config.py)

---

> **왜 "레벨"에서 "단위"로 바꿨나** — 예전 글은 맨 앞 Lv.0에 메모리·커널·Pallas·online
> softmax·sink의 직관을 한꺼번에 몰아넣었습니다. 그래서 online softmax가 정작 필요해지기도
> 전에 너무 일찍, 동떨어진 채 튀어나왔죠. 이 노트북에서 online softmax는 **D에서 key 타일을
> 스트리밍하기 시작하는 바로 그 지점**에 직관 → 수식 → 정확성 증명까지 한 단위로 모여
> 있습니다. 나머지 개념도 같은 원칙으로 "쓰이는 자리"에 배치했습니다.

> **실행 안내**: 모든 `[실습]` 셀은 이 저장소 루트에서 검증되었습니다. CPU `interpret` 모드로 돌며 TPU가 필요 없습니다. 노트북을 **저장소 루트**에서 띄우거나, 바로 아래 셀에서 루트를 `sys.path`에
> 추가하세요. 실습 셀은 위에서 아래로 순서대로 실행하면 앞 셀의 변수 `q`, `K_comp` 등을
> 이어서 씁니다.

In [ ]:
# 저장소 루트에서 실행 중이 아니라면 아래 한 줄의 주석을 풀고 경로를 맞춰주세요.
import sys, os
# sys.path.insert(0, "/path/to/auto-jax-kernel")   # dsv4 패키지를 임포트할 수 있게

import jax, jax.numpy as jnp
print("jax", jax.__version__, "| devices:", jax.devices())

---

# A. 왜 이 커널이 필요한가

여기선 코드를 거의 안 봅니다. 대신 뒤에서 코드를 읽을 때 필요한 **두 개의 머릿속 그림**을
만듭니다: ① 무엇이 아픈가(메모리), ② 하드웨어가 어떻게 생겼나(HBM/VMEM). 이 둘이 잡히면
"커널을 왜 저렇게 짜야 하는지"가 저절로 보입니다.

## A.1 왜 "그냥 JAX"로 하면 안 되나? — 메모리 통증

평범한 attention을 JAX로 짜면 이렇게 됩니다:

```python
logits = q @ K.T            # [n_h, S]  ← 모든 query head × 모든 key
weights = softmax(logits)   # [n_h, S]
out = weights @ V           # [n_h, c]
```

문제는 가운데 `logits`/`weights` 입니다. key가 $S$개면 이 행렬이 $n_h \times S$ 크기로 **통째로
메모리에 올라갑니다**. $S$가 수천~수만이면 이게 엄청 커지고, 특히 TPU의 빠른 메모리인 VMEM엔 안 들어갑니다.

> **핵심 문제**: 중간 결과인 attention 가중치 행렬이 너무 크다.
> FlashAttention은 **이 중간 행렬을 아예 안 만드는** 기법입니다. 그 답인 online softmax는 §D에서 정확히 필요해지는 자리에 나옵니다. 지금은 "큰 중간 행렬이
> 아프다"만 기억하세요.

## A.2 TPU 한 장의 구조 — HBM vs VMEM

TPU 칩 하나를 아주 단순화하면 이렇습니다:

```
┌─────────────────────────────────────────────┐
│  TPU 칩                                      │
│                                              │
│   ┌──────────┐        ┌──────────────────┐   │
│   │  VMEM    │ ←빠름→  │  연산 유닛          │  │
│   │ (작음,   │         │  - MXU: 행렬곱     │   │
│   │  ~수십MB)│         │  - VPU: 벡터연산    │  │
│   └────┬─────┘        └──────────────────┘   │
│        │ 느림 (여기 왕복이 비쌈!)                 │
│   ┌────┴─────────────────────────────────┐   │
│   │  HBM (큼, ~수십GB, 하지만 느림)           │  │
│   └──────────────────────────────────────┘  │
└─────────────────────────────────────────────┘
```

- **HBM** (High Bandwidth Memory): 크지만 **느립니다**. 모든 입력/출력 텐서가 여기 삽니다.
- **VMEM** (Vector Memory): 작지만 **빠릅니다**. 연산 유닛이 실제로 계산할 때 데이터는 여기
  올라와 있어야 합니다.
- **MXU**: 행렬곱 전용 하드웨어. bf16 입력을 받아 fp32로 누적하는 경로가 빠름.
- **VPU**: 벡터 연산 유닛. 마지막 차원이 레인폭 **128의 배수**일 때 효율적 — 나중에 "왜 `BS`는 128의 배수여야 하나" 규칙의 원천.

> **성능의 제1법칙**: HBM ↔ VMEM 왕복을 줄여라.
> 큰 중간 행렬을 HBM에 썼다 읽었다 하면 느립니다. **데이터를 VMEM에 올린 김에 최대한 많이
> 계산하고** 결과만 HBM에 쓰는 게 빠릅니다. 커널 최적화는 거의 다 이 이야기입니다.

## A.3 그래서 목표는

위 둘을 합치면 이 커널이 따라야 할 두 가지 규칙이 나옵니다:

1. **큰 중간 행렬인 attention 가중치를 아예 만들지 않는다.** A.1의 문제입니다.
2. **데이터를 VMEM에 올린 김에 최대한 계산하고, HBM에는 결과만 쓴다.** A.2의 제1법칙입니다.

이 둘을 가능하게 해주는 도구가 §B의 **Pallas**, 이 둘을 실제로 달성하는 기법이 §D의
**online softmax**입니다. 순서가 중요합니다. 먼저 §B–C에서 도구로 타일을 흘려보내는 골격을
세우고, 그 골격 위에서 §D에서 online softmax가 왜 필연인지 보게 됩니다.

---

# B. Pallas 도구 — 타일로 쪼개 도는 법

§A의 두 규칙을 코드로 옮기려면 "큰 배열을 작은 타일로 쪼개 같은 함수를 반복 실행"하는
장치가 필요합니다. 그게 Pallas입니다.

## B.1 커널과 Pallas

**커널**은 작은 데이터 조각 하나, 즉 타일에 대해 무슨 계산을 할지 적은 함수입니다.
**Pallas**는 JAX에서 그 커널을 파이썬으로 작성하게 해주는 도구입니다.

동작 원리는 이렇습니다. 큰 배열을 통째로 다루지 않고 작은 타일로 쪼갠 뒤, 같은 커널 함수를
격자 `grid`의 각 칸마다 한 번씩 반복 실행합니다.

```
        grid = 격자. 커널을 몇 번 반복할지
        ┌─────────────────────────────────────────────────┐
HBM →   │ 타일을 VMEM에 올림 → [커널 함수] → 결과 타일을 HBM에 씀 │   → HBM
        └─────────────────────────────────────────────────┘
              이 과정이 grid의 칸 수만큼 반복됨
```

- 각 실행은 자기 타일만 봅니다. 타일은 자동으로 빠른 VMEM에 올라옵니다.
- 우리가 정의할 것은 셋입니다. ① 격자를 몇 칸으로 나눌지 `grid`, ② 각 칸이 전체의 어느
  조각을 볼지 `BlockSpec`, ③ 조각 하나에 대한 계산인 커널 함수.

## B.2 BlockSpec / index_map — "이 칸은 어느 조각을 보나"

격자의 칸마다 "전체 배열의 어느 부분을 가져올지" 알려줘야 합니다. 그게 `BlockSpec`입니다.

```python
pl.BlockSpec(block_shape, index_map)
#            └ 타일 모양   └ (격자 인덱스) → 블록 좌표
```

`index_map`은 격자 인덱스를 받아 "이 칸은 몇 번째 블록을 가져올지"를 반환하는 함수입니다.
**주의**: 반환값은 원소 좌표가 아니라 **블록 좌표**입니다. 좌표 `2`는 원소
`2 × block_shape` 위치를 가리킵니다.

지금은 "각 칸이 큰 배열에서 자기 몫의 작은 타일을 받아온다" 정도만 기억하면 됩니다. 구체적인
예는 §D.4에서 실제 커널의 `BlockSpec`을 한 줄씩 읽습니다.

## B.3 [실습] 환경 준비 + 가장 작은 호출

이제 실제로 돌려봅시다. TPU는 필요 없고 CPU에서 시뮬레이션됩니다. 아직 커널 내부는
**블랙박스**로 두고, 순수 JAX 레퍼런스와 같은 답을 내는지만 확인합니다. 다음 셀들에서 이
블랙박스를 하나씩 열어볼 겁니다.

In [ ]:
import jax, jax.numpy as jnp
from dsv4 import eager
from dsv4.kernel import sparse_attn_kernel   # 공개 표면 (기본 V2로 라우팅)

# 아주 작은 문제. B=배치1, n=query8개, head4개, head차원128
B, n, n_h, c = 1, 8, 4, 128
n_blk, k, n_win = 4, 3, 4    # 압축블록 4개, query당 top-3, SWA 윈도우 4
ks = jax.random.split(jax.random.PRNGKey(0), 3)
q         = jax.random.normal(ks[0], (B, n, n_h, c))
K_comp    = jax.random.normal(ks[1], (B, n_blk, c))
K_swa     = jax.random.normal(ks[2], (B, n, n_win, c))
attn_sink = jnp.zeros((n_h,))
# query마다 고른 압축블록 인덱스. -1 = "고를 게 부족함"(패딩)
topk_idxs = jnp.array([[[0,1,2],[1,2,-1],[0,-1,-1],[2,3,0],
                        [1,1,2],[0,2,3],[3,-1,-1],[0,1,3]]], dtype=jnp.int32)

out_kernel = sparse_attn_kernel(q, K_comp, topk_idxs, K_swa, attn_sink)
out_eager  = eager.sparse_attn_with_sink(q, K_comp, topk_idxs, K_swa, attn_sink)

print("출력 모양:", out_kernel.shape)                       # (1, 8, 4, 128)
print("레퍼런스와 차이:", float(jnp.max(jnp.abs(out_kernel - out_eager))))  # ~3e-7

**관찰 포인트**:
- 출력은 `[B, n, n_h, c]` = `(1, 8, 4, 128)`. query 토큰마다, head마다, 길이-`c` 벡터 하나.
- 커널 결과가 순수 JAX 레퍼런스 `eager`와 `~3e-7`로 거의 같습니다. 즉 **커널은 레퍼런스를
  빠르게 다시 계산하는 것**일 뿐, 다른 답을 내지 않습니다. 레퍼런스가 항상 정답 기준입니다.

> 🎯 **연습 B-A**: `attn_sink`를 `jnp.zeros`에서 큰 값인 `jnp.full((n_h,), 5.0)`으로 바꿔
> 다시 돌려보세요. 출력 노름 `jnp.linalg.norm(out_kernel)`이 **작아집니다**. 왜일까요?
> 힌트는 §C.2의 sink입니다. sink가 크면 분모가 커져 모든 가중치가 깎입니다.

이제 이 결과가 **커널 내부에서 어떻게 만들어지는지** 봅니다. 먼저 §C에서 무엇을 계산하는지, 그다음 §D에서 어떻게 계산하는지 봅니다.

---

# C. 무엇을 계산하나 — sparse MQA + sink

커널 내부를 열기 전에, 이 커널이 정확히 **어떤 수식**을 계산하는지부터 못 박습니다.
"어떻게"(online softmax, 타일링)는 모두 이 식을 *값을 안 바꾸면서* 빠르게 구하는 방법일
뿐입니다.

## C.1 우리가 계산하려는 식

head $h$, query $t$에 대해:

$$
\boxed{\;
o_{t,h} = \frac{\displaystyle\sum_{s\in \text{valid}} e^{\,\text{scale}\cdot q_{t,h}\cdot k_s}\; k_s}
{\displaystyle\sum_{s\in \text{valid}} e^{\,\text{scale}\cdot q_{t,h}\cdot k_s} \;+\; e^{\text{sink}_h}}
\;}
$$

평범한 attention과 모양이 같되, 분자의 value 자리에 $k_s$가 오고(아래 V==K), 합이 "고른
key(valid)"에 대해서만 돌며, 분모에 $e^{\text{sink}_h}$가 더 붙습니다. 이 세 가지가 이
커널의 특이점입니다.

## C.2 세 가지 특이점

**① MQA + V==K.** query는 head가 `n_h`개지만 key/value는 head 하나를 공유하는 Multi-Query
Attention이고, **value = key**입니다. 논문 §2.3의 V4 규약대로 각 압축 KV 엔트리가 key이자
value입니다. 그래서 커널에 들어오는 건 `K_full` 하나뿐이고, PV 곱에서도 `k`를 value로 그대로
씁니다. 별도 value 텐서 `V`는 어디에도 없습니다.

**② Sparse.** key 집합 = `[top-k 압축블록 | SWA 윈도우]`. 즉 query마다 인덱서가 고른 중요한
압축블록 `k`개와 직전 `n_win`개 토큰의 슬라이딩-윈도우 키를 **이어붙인** 것이 그 query가 보는
전부입니다. 이걸 고르고 잇는 일은 §D.3의 순수 JAX가 하고, 커널은 이미 고른 것에 대한
attention만 합니다.

**③ Attention sink.** 분모에만 더해지는 $e^{\text{sink}_h}$. 여기서 한 단위로 끝까지 봅니다.

head마다 학습되는 스칼라 $\text{sink}_h$가 하나 있습니다. value가 0인 가짜 key 하나라고 생각하면 정확합니다.

$$
o = \frac{\sum_s e^{\text{점수}_s}\, v_s \;+\; e^{\text{sink}_h}\cdot \mathbf{0}}
{\;\sum_s e^{\text{점수}_s} \;+\; e^{\text{sink}_h}\;}
$$

분자에는 $e^{\text{sink}_h}\cdot 0 = 0$이 보태져 **아무것도 안 더하고**, 분모에는
$e^{\text{sink}_h}$가 보태져 **모든 softmax 가중치를 동시에 일정 비율로 깎습니다**. 강하게 볼
key가 없다는 상태를 이 스칼라 하나로 표현하는 셈이고, 그래서 연습 B-A에서 sink를 키우면 출력이
작아졌습니다. 구현상 이 가짜 key는 매 스텝 끼지 않고 **마지막에 한 번** 결합됩니다. 그 코드는
§D.6에서 봅니다.

---

# D. V1 순전파 — FlashAttention-with-sink 한 줄씩

목표: `sparse_attn_kernel`을 부르면 안에서 벌어지는 일을 한 줄씩 이해하기.
파일은 [`kernel_v1.py`](./kernel_v1.py).

## D.1 순전파 호출 트리

```
sparse_attn_kernel_v1()              진입점: config로 블록크기 결정
  └─ _v1_forward()                   gather + 패딩 + custom_vjp 호출
       ├─ _gather_concat_mask()      K_full / mask 만들기, 순수 JAX
       └─ _make_flash_diffable()     custom_vjp 경계
            └─ _flash_attn_with_sink_pallas()   Pallas 호출, grid/BlockSpec
                 └─ _flash_sink_kernel()         ★ TPU에서 도는 진짜 커널
```

맨 안쪽 `_flash_sink_kernel`이 별이고, 나머지는 데이터를 깎아주는 껍데기입니다. 아래에서
**바깥 껍데기부터 안쪽 별까지** 순서대로 벗겨봅니다.

## D.2 입력 모양과 기호

| 기호 | 뜻 | 실습값 | Flash 프리셋 |
|------|------|------|------|
| `B` | 배치 | 1 | 작음 |
| `n` | query 토큰 수 | 8 | 큼 |
| `n_h` | query head 수 | 4 | 64 |
| `c` | head 차원 | 128 | 512 |
| `k` | query당 top-k | 3 | 512 |
| `n_win` | SWA 윈도우 | 4 | 128 |
| `S` | `k+n_win` (패딩 후 `BS`의 배수) | 7→8 | ~640 |
| `BQ` | 한 프로그램이 맡는 query 수 | config | config |
| `BS` | 한 스텝이 먹는 key 수 | config | config |

## D.3 껍데기 1: gather와 패딩

### `_gather_concat_mask` (kernel_v1.py 439~460행) — key 집합 만들기

```python
safe_idx = jnp.where(topk_idxs < 0, 0, topk_idxs)        # -1을 0으로. 안전 인덱스
K_sel = jnp.take_along_axis(K_comp[:,None].repeat(n,1),  # query마다 고른 압축블록 모음
                            safe_idx[...,None].repeat(c,-1), axis=2)   # [B,n,k,c]
K_full = jnp.concatenate([K_sel, K_swa], axis=2)         # [B,n,k+n_win,c]
mask = jnp.concatenate([topk_idxs >= 0,                  # top-k: -1 아니면 유효
                        jnp.ones((B,n,n_win), bool)],    # SWA: 전부 유효
                       axis=-1)                          # [B,n,k+n_win]
return K_full, mask
```

- 부족 패딩인 `-1`을 `0`으로 바꿔 안전하게 gather한 뒤, **mask로 그 자리를 무효 처리**합니다.
- `K_full`: 한 query가 보는 key 전체 = `[고른 압축블록 | SWA]`. §C.2의 sparse 집합입니다.
- `mask`: 같은 위치가 유효한지 여부.
- **이 함수는 순수 JAX**라 역전파는 자동미분이 처리하며, 이는 §E.4에서 설명합니다. 그래서 커널은 gather를 몰라도 됩니다.

### `_v1_forward`의 패딩 (kernel_v1.py 518~548행)

```python
pad_n = (-n) % config.bq   # n을 BQ의 배수로
pad_s = (-S) % config.bs   # S를 BS의 배수로
```

격자가 query를 `BQ`개씩, key를 `BS`개씩 깔끔히 나눠야 하므로 모자란 만큼 가짜를 채웁니다.
`(-x) % b`는 x를 b의 배수로 올릴 때 모자란 칸 수를 줍니다. 패딩된 자리는 `mask=False`라
계산에 영향이 0이고, 끝나면 `out[:, :n]`으로 잘라냅니다.

## D.4 껍데기 2: Pallas 호출 (`_flash_attn_with_sink_pallas`, 233~294행)

### 격자

```python
nsteps = S // bs                 # key를 BS개씩 → 몇 스텝?
grid = (B, n // bq, nsteps)      # (배치, query타일, key스텝)
```

```
grid = (B, n//BQ, nsteps)
        │   │       └ 축2: key를 BS개씩 순차. 누적, online softmax
        │   └──────── 축1: query를 BQ개씩. 병렬 가능
        └──────────── 축0: 배치. 병렬 가능
```

한 `(배치, query타일)`에 대해 커널이 `s = 0,1,...,nsteps-1`을 차례로 돌며 §D.5에서 설명할
online softmax를 굴립니다.

### BlockSpec (각 칸이 보는 조각)

```python
q_spec    = pl.BlockSpec((1, bq, n_h, c), lambda b,qi,si: (b, qi, 0, 0))  # si에 안 변함→재사용
k_spec    = pl.BlockSpec((1, bq, bs,  c), lambda b,qi,si: (b, qi, si, 0)) # si마다 다른 조각!
mask_spec = pl.BlockSpec((1, bq, bs),     lambda b,qi,si: (b, qi, si))
sink_spec = pl.BlockSpec((n_h,),          lambda b,qi,si: (0,))           # 항상 전체
o_spec    = pl.BlockSpec((1, bq, n_h, c), lambda b,qi,si: (b, qi, 0, 0))
lse_spec  = pl.BlockSpec((1, bq, n_h, 1), lambda b,qi,si: (b, qi, 0, 0))
```

읽는 법: **query는 key 스텝 `si`가 변해도 그대로**라 한 query 타일을 모든 key 스텝 동안
재사용하고, **key/mask만 `si`에 따라 다른 조각**을 흘려보냅니다. 이것이 §B.1에서 말한
"타일을 쪼개 격자의 각 칸에 반복 실행"이 실제로 도는 방식입니다.

### scratch (누적 상태 보관함)

```python
scratch_shapes=[
    pltpu.VMEM((1, bq, n_h, c), jnp.float32),  # m
    pltpu.VMEM((1, bq, n_h, c), jnp.float32),  # l
    pltpu.VMEM((1, bq, n_h, c), jnp.float32),  # acc
]
```

key 스텝들 사이에 값이 유지되는 VMEM. 여기 담길 $m, \ell, \text{acc}$ 세 개가 바로 §D.5
online softmax가 누적해 가는 상태입니다. 이것이 무엇이고 왜 셋이면 충분한지는 바로 다음
단위에서 봅니다.

> **레이아웃 미묘함**: $m, \ell$은 원래 query·head당 스칼라 `[..,n_h,1]`이지만, `n_h`가 레인폭
> 128의 배수가 아닐 수 있어 정렬 규칙에 걸립니다. 그래서 일부러 `c` 차원을 붙여 `[..,n_h,c]`로
> 만들고 스칼라를 `c` 레인 전체에 복사합니다. 읽을 땐 `[...,:1]`로 레인 0만 뗍니다. 메모리를
> 조금 더 쓰는 대신 정렬 문제를 피합니다.

### 격자 실행 흐름 — 한 장의 타임라인

위 세 가지 grid·BlockSpec·scratch가 실제로 어떻게 맞물려 도는지 한 장으로 봅시다.
먼저 **바깥 두 축** `(b, qi)`은 서로 독립이라 병렬로 흩뿌려집니다:

```
바깥 격자  (b, qi) = B × (n/BQ) 개의 "독립 프로그램"
─────────────────────────────────────────────────────────
   (0,0) (0,1) (0,2) ...        각 칸은 서로 안 섞임 → "parallel"
   (1,0) (1,1) (1,2) ...        → 메가코어 v5p/v6p면 두 코어로 자동 분할
     │                          → 각 칸이 아래 "안쪽 타임라인"을 독립적으로 1회 수행
     └─ 이 칸 하나를 확대하면 ↓
```

그 한 칸 `(b, qi)` 안에서, **안쪽 축** `si`(key 스텝)는 순차적으로 돌며 누적합니다:

```
한 (b, qi) 안에서  si = 0 ─→ 1 ─→ 2 ─→ … ─→ nsteps-1   순차·누적

  si =          0             1             2       …      nsteps-1
           ┌──────────┐  ┌──────────┐  ┌──────────┐     ┌───────────┐
 로드       q[b,qi]       q[b,qi]       q[b,qi]            q[b,qi]      ← si 무관! 재사용
 HBM→VMEM   k[b,qi, 0]    k[b,qi, 1]    k[b,qi, 2]   …    k[b,qi,last] ← si마다 새 타일
            mask[.., 0]   mask[.., 1]   mask[.., 2]       mask[..,last]
           └────┬─────┘  └────┬─────┘  └────┬─────┘     └─────┬─────┘
                ▼             ▼             ▼                  ▼
 커널      @when(s==0)     QK^T·scale    QK^T·scale          QK^T·scale
 동작      init m,l,acc    → mask        → mask        …     → mask
            (−1e30,0,0)    → softmax     → softmax           → softmax
                │          m,l,acc 갱신  m,l,acc 갱신         m,l,acc 갱신
                │             │             │            @when(s==last)
                │             │             │            sink 결합 → out, lse
                ▼             ▼             ▼                  ▼
 scratch   ╔═══════════════════════════════════════════════════════════╗
 (VMEM)    ║  m, l, acc : 스텝을 가로질러 계속 유지됨 (HBM 안 거침!)       ║
           ╚═══════════════════════════════════════════════════════════╝
 출력                                                       out[b,qi], lse[b,qi]
 VMEM→HBM                                                   ← 마지막 스텝에 딱 1번만 씀
```

이 그림이 §A.2 제1법칙, 즉 HBM 왕복 줄이기의 실현입니다.
- **q는 한 번만 로드해 `nsteps`번 재사용**. `si`에 따라 안 변하는 BlockSpec 덕분.
- **k/mask만 스트리밍**. `si`마다 새 타일을 받고 옛 타일은 버림.
- **중간 상태 `m,l,acc`는 VMEM에 눌러두고 누적** — 큰 attention 가중치 행렬을 HBM에 절대 안 만듦.
- **HBM에 쓰는 건 마지막에 `out`/`lse` 한 번뿐.**

자, 골격이 다 섰습니다: key 타일이 흘러오고, 우리는 `m,l,acc` 세 개만 들고 누적합니다.
그런데 **점수 행렬 전체를 안 만들고** 어떻게 softmax가 정확할 수 있을까요? 그 답이 다음
단위입니다.

## D.5 ★ online softmax — 한 단위로 끝까지

§A.1의 문제는 중간 attention 가중치 행렬이 너무 크다는 것이었습니다. §D.4의 격자는 key를 `BS`개씩
타일로 흘려보내고 scratch에 `m, l, acc` 세 개만 들고 있었죠. **그 세 개만으로 전체 점수 행렬을
한 번도 만들지 않고** softmax를 정확히 끝내는 기법 — 그게 online softmax입니다. 직관 → 수식 →
정확성 증명까지 여기서 한 번에 봅니다.

### 아이디어: 3개의 작은 누적값만

key를 한 번에 다 보지 않고 타일 한 묶음씩 흘려보내면서, 딱 3개의 작은 누적값만 들고
다닙니다.

| 누적값 | 뜻 |
|--------|------|
| $m$ | 지금까지 본 점수의 **최댓값** |
| $\ell$ | softmax **분모**. $\sum e^{x-m}$의 부분합 |
| $\text{acc}$ | softmax **분자**. 가중합된 value $\sum e^{x-m} v$의 부분합 |

새 묶음이 올 때마다 이 3개를 조금씩 갱신합니다. 그러면 **전체 점수 행렬을 절대 한꺼번에 안
들고도** 정확한 softmax 결과를 얻습니다. 메모리는 묶음 하나 크기면 충분 — 이게 §D.4 scratch가
`m,l,acc` 셋뿐이었던 이유입니다.

### 왜 최댓값 $m$을 추적하나 — overflow 방지

$e^{\text{점수}}$를 그냥 계산하면 점수가 조금만 커도 $e^{100}$처럼 값이 폭주합니다. 그래서
항상 최댓값을 빼고 $e^{\text{점수}-m}$으로 계산합니다. 분자와 분모에서 $e^{-m}$이 약분되므로
결과는 같고 수치적으로 안전합니다. 문제는 타일을 흘려보내는 동안 최댓값이 **계속 바뀐다**는
것이고, 그걸 보정하는 게 아래 $\alpha$입니다.

### 갱신 규칙

새 묶음이 와서 최댓값이 $m_{\text{prev}} \to m_{\text{new}}$로 바뀌면, 과거에 모은
$\ell, \text{acc}$를 $\alpha = e^{m_{\text{prev}} - m_{\text{new}}}$배로 깎아 기준을 맞춘 뒤
새 몫을 더합니다:

$$
m_{\text{new}} = \max(m_{\text{prev}}, m_{\text{curr}}),\quad
p_s = e^{\text{logit}_s - m_{\text{new}}},\quad
\alpha = e^{m_{\text{prev}}-m_{\text{new}}}
$$
$$
\ell_{\text{new}} = \alpha\,\ell_{\text{prev}} + \sum_s p_s,\qquad
\text{acc}_{\text{new}} = \alpha\,\text{acc}_{\text{prev}} + \sum_s p_s\, v_s
$$

직관: 기준 최댓값이 내려갔으니 과거 누적을 $\alpha$배로 깎아 같은 기준으로 맞춘 뒤 새 묶음
몫을 더합니다. $\alpha \le 1$이라 과거를 깎는다고 표현했습니다. 마지막에 $\text{acc}/\ell$을
나누면 공통 인자 $e^{-m}$이 약분돼 정답이 나옵니다. §D.6 마지막 스텝에서 sink와 함께
처리합니다.

### 왜 정확한가 — 증명

스텝 $j=1..N$의 점수 집합 $L_j$, $x_s=\text{logit}_s$. 목표:
$o = \dfrac{\sum_j\sum_{s\in L_j} e^{x_s} v_s}{\sum_j\sum_{s\in L_j} e^{x_s}}$.

스텝 $t$까지 처리한 뒤의 **불변량**:

$$
m^{(t)}=\max_{j\le t, s} x_s,\quad
\ell^{(t)}=\sum_{j\le t,s} e^{x_s-m^{(t)}},\quad
\text{acc}^{(t)}=\sum_{j\le t,s} e^{x_s-m^{(t)}} v_s
$$

최댓값이 $m^{(t)}\to m^{(t+1)}$로 바뀌면 옛 항은 $e^{x_s-m^{(t)}}$로 저장돼 있으니
$\alpha=e^{m^{(t)}-m^{(t+1)}}$를 곱해 기준을 맞춥니다:

$$
e^{x_s-m^{(t+1)}}=\alpha\,e^{x_s-m^{(t)}}
\;\Rightarrow\;
\ell^{(t+1)}=\alpha\ell^{(t)}+\textstyle\sum_{L_{t+1}}p_s,\quad
\text{acc}^{(t+1)}=\alpha\,\text{acc}^{(t)}+\textstyle\sum_{L_{t+1}}p_s v_s
$$

이게 위 갱신 코드 그대로입니다. 마지막에 $\text{acc}^{(N)}/\ell^{(N)}$에서 공통 $e^{-m^{(N)}}$가
약분돼 정답 $o$가 됩니다. sink는 $v=0$인 항을 분모 $\ell$에만 추가합니다. $\blacksquare$

> **핵심 한 줄**: 타일을 어떻게 쪼개든, 즉 `BS`/`nsteps`를 어떻게 잡든 이 갱신은 같은 답으로
> 수렴합니다. 블록 크기는 속도와 메모리 손잡이일 뿐 정답을 바꾸지 않습니다. §D.8 실습에서
> 눈으로 확인합니다.

## D.6 ★ 커널 본체 `_flash_sink_kernel` (148~226행)

이제 §D.5의 갱신을 실제 커널 코드로 봅니다. 이 함수가 격자의 각 `(b, qi, si)`에서 한 번씩
실행됩니다. `_ref` 인자들은 VMEM 타일 참조.

### 첫 스텝 초기화

```python
s = pl.program_id(2)                 # 지금 몇 번째 key 스텝?
@pl.when(s == 0)
def _init_scratch():
    m_ref[...]   = jnp.full(m_ref.shape, _NEG_INF_F32)  # -1e30. 사실상 -∞이지만 유한
    l_ref[...]   = jnp.zeros(l_ref.shape)
    acc_ref[...] = jnp.zeros(acc_ref.shape)
```

- `@pl.when(조건)`: 조건일 때만 실행하는, 파이썬 `if`의 트레이싱 버전.
- $m$을 진짜 `-inf`가 아니라 `-1e30`으로 둡니다. 첫 블록이 통째로 mask되면
  `exp(-inf - -inf) = nan`이 터지는데, 유한한 큰 음수면 `exp(거대음수)=0`이라 안전합니다.

### QK^T → 점수

```python
q = q_ref[...].astype(jnp.float32)
k = k_ref[...].astype(jnp.float32)
mask = mask_ref[...].astype(jnp.bool_)
logits = _dot2(q, k, contract=(3, 3)) * jnp.float32(scale)   # [1,BQ,n_h,BS]
```

$\text{logit}_s = \text{scale}\cdot(q\cdot k_s)$. `_dot2`는 head 차원 `c`를 내적으로 줄이고
`n_h`와 `BS`를 남겨, 각 head가 각 key에 매긴 점수를 만듭니다. 이 과정에서 MQA가 자연히
처리되며, `_dot2`는 §D.7에서 다룹니다.

### mask 적용

```python
mask_b = mask[:, :, None, :]                              # head 축 끼워넣어 브로드캐스트
logits = jnp.where(mask_b, logits, jnp.float32(_NEG_INF_F32))  # 무효 위치 = -1e30
```

무효 key의 점수를 `-1e30`으로 → `exp`에서 거의 0 → 가중치 0 → "없는 key".

### online softmax 갱신 — §D.5의 수식 그대로

```python
m_prev, l_prev = m_ref[...,:1], l_ref[...,:1]   # 레인 0만. 스칼라
acc_prev = acc_ref[...]

m_curr = jnp.max(logits, axis=-1, keepdims=True)   # 이 블록 최댓값
m_new  = jnp.maximum(m_prev, m_curr)               # 갱신된 누적 최댓값

p = jnp.exp(logits - m_new)                        # 새 블록 기여분
p = jnp.where(mask_b, p, 0.0)                       # 무효는 정확히 0
alpha = jnp.exp(m_prev - m_new)                    # 과거 보정 계수, ≤1
l_new = alpha * l_prev + jnp.sum(p, axis=-1, keepdims=True)   # 분모 갱신

pv = _dot2(p, k, contract=(3, 2))                  # Σ p_s·v_s. V==K라 k 사용
acc_new = acc_prev * alpha + pv                    # 분자 갱신

m_ref[...]   = jnp.broadcast_to(m_new, m_ref.shape)   # 다시 c 레인에 복사 저장
l_ref[...]   = jnp.broadcast_to(l_new, l_ref.shape)
acc_ref[...] = acc_new
```

§D.5의 $m_{\text{new}}, p_s, \alpha, \ell_{\text{new}}, \text{acc}_{\text{new}}$가 한 줄씩
그대로 대응됩니다. 여기까지가 **한 key 스텝**. `s=0…nsteps-1` 동안 반복되며 누적됩니다.

### 마지막 스텝: sink 결합 + 출력 + lse

```python
@pl.when(s == nsteps - 1)
def _finalize():
    m_final, l_final = m_ref[...,:1], l_ref[...,:1]
    acc_final = acc_ref[...]
    sink_b = sink_ref[...][None,None,:,None]            # [1,1,n_h,1]

    m_combined  = jnp.maximum(m_final, sink_b)
    alpha_final = jnp.exp(m_final - m_combined)
    sink_term   = jnp.exp(sink_b - m_combined)          # 가짜 key의 분모 기여
    denom       = l_final * alpha_final + sink_term
    out         = (acc_final * alpha_final) / denom     # value=0이라 분자엔 sink 없음
    o_ref[...]  = out.astype(o_ref.dtype)

    lse = m_combined + jnp.log(denom)                   # log-sum-exp. 역전파용 저장
    lse_ref[...] = lse.astype(jnp.float32)
```

- §C.2의 sink를 value가 0인 가짜 key로 보고 **마지막에 한 번** 결합합니다. 매 스텝 끼지 않는
  이유는 분모에만 영향을 줘서 끝에 한 번이면 충분하기 때문입니다.
- `lse` = $\log(\sum e^{\text{logit}} + e^{\text{sink}})$. 순전파엔 안 쓰지만 **역전파에서 `p`를
  한 방에 복원**하려고 저장합니다. §E에서 씀.

## D.7 `_dot2` — 왜 특수 행렬곱이 필요한가 (119~141행)

커널 안 모든 행렬곱은 `_dot2`를 씁니다.

```python
def _dot2(lhs, rhs, *, contract):
    lc, rc = contract
    b0, b1 = lhs.shape[0], lhs.shape[1]
    out = jax.lax.dot_general(
        lhs.reshape((b0*b1,)+lhs.shape[2:]), rhs.reshape((b0*b1,)+rhs.shape[2:]),
        dimension_numbers=(((lc-1,),(rc-1,)), ((0,),(0,))),
        preferred_element_type=jnp.float32)
    return out.reshape((b0, b1)+out.shape[1:])
```

**문제**: 우리 타일은 앞 두 축 `(B=1, BQ)`이 둘 다 배치인데, TPU `tpu.matmul`은 배치 축을
**1개만** 지원합니다.
**해결**: 두 배치 축을 곱해 `reshape`로 하나로 접고, `c`나 `BS`를 `contract` 축으로 내적해
줄인 뒤 도로 폅니다. `b0==1`이라 이 접기와 펴기는 레이아웃 변화가 없어 비용이 0입니다.
`preferred_element_type=f32` 덕분에 bf16 입력도 fp32 MXU 경로로 누적합니다.

## D.8 [실습] 블록 크기를 직접 바꿔보기

§D.5의 "핵심 한 줄"을 검증합니다. `config`로 블록 크기를 직접 정해 `nsteps`가 어떻게 변하고,
그래도 **답이 같은지** 봅니다.

In [ ]:
from dsv4.kernel_config import KernelConfig
from dsv4.kernel_v1 import sparse_attn_kernel_v1
# (B.3 실습의 q, K_comp, topk_idxs, K_swa, attn_sink, out_eager 를 이어서 사용)

for bs in (4, 2):
    cfg = KernelConfig(bq=2, bs=bs, lane_size=1, interpret=True)  # lane_size=1: 작은 c 허용
    o = sparse_attn_kernel_v1(q, K_comp, topk_idxs, K_swa, attn_sink, config=cfg)
    S = k + n_win                       # 3 + 4 = 7
    pad_s = (-S) % bs                   # 4의 배수→8, 2의 배수→8
    print(f"bs={bs}: S={S}->{S+pad_s}, nsteps={(S+pad_s)//bs}, "
          f"diff={float(jnp.max(jnp.abs(o-out_eager))):.1e}")
# bs=4: S=7->8, nsteps=2, diff=2.4e-07
# bs=2: S=7->8, nsteps=4, diff=2.4e-07

**관찰**: `bs`를 4→2로 줄이면 같은 key를 더 잘게 나눠 더 여러 번 흘려보내므로 `nsteps`가
2→4로 **늘어납니다**. 그런데 **결과는 동일**합니다. 이것이 §D.5 online softmax의 핵심입니다.
**타일을 어떻게 쪼개도 정답이 같습니다.**

> 🎯 **연습 D-A**: `bq=2`를 `bq=4`, `bq=8`로 바꿔보세요. 결과가 같은지 확인합니다. query 타일
> 크기도 정답을 바꾸지 않습니다.
> 🎯 **연습 D-B**: `topk_idxs`의 한 행을 전부 `-1`로 바꿔보세요. 예를 들어 3번 query를
> `[-1,-1,-1]`로. 그 query는 SWA만 보게 됩니다. §D.6 `-1e30` 트릭 덕분에 NaN 없이 도는지 확인합니다.

---

# E. V1 역전파 — 저장한 lse로 gradient 되돌리기

순전파를 이해했으니 이제 학습에 필요한 **gradient**를 봅니다. 같은 파일
[`kernel_v1.py`](./kernel_v1.py) 318~432행.

## E.1 역전파가 뭘 구하나 + 두 핵심 아이디어

순전파가 `out = f(q, K_full, mask, sink)`라면, 역전파는 위에서 내려온 gradient `dout`을 받아
**각 입력에 대한 gradient** `dq, dK_full, dsink`를 구합니다. mask는 미분 대상이 아니므로
`dmask`는 0입니다. 이를 chain rule로 위층에 계속 전달해 학습이 됩니다.

핵심 아이디어 두 가지:

**1. 재계산.** 순전파의 attention 가중치 `p`를 저장해두면 메모리를 많이 먹습니다. 대신
**저장한 `lse` 하나**로 `p`를 다시 만듭니다. $p_s = e^{\text{logit}_s - \text{lse}}$이고
logit은 $q\cdot k$로 다시 계산합니다. §D.6에서 `lse`를 저장한 이유가 이것입니다.

**2. softmax 역전파 항등식.** 아래 한 줄이 전부입니다.

$$
d\text{logit}_s = p_s\,(dp_s - D),\qquad dp_s = dout\cdot v_s,\qquad
D = \sum_c dout\cdot o = dout\cdot o
$$

왜 $D = dout\cdot o$? $o = \sum_s p_s v_s$이고 softmax 미분을 정리하면 공통항
$D = \sum_s p_s\,dp_s = dout\cdot(\sum_s p_s v_s) = dout\cdot o$가 떨어집니다. 그래서 $D$를
미리 한 번 계산해 둡니다.

## E.2 사전 계산: D와 dsink (374~432행, JAX 부분)

```python
# D = Σ_c (dout · o).  shape [B,n,n_h,1], lse와 같은 레이아웃이라 같은 BlockSpec으로 들어감
D = (dout.astype(f32) * out.astype(f32)).sum(axis=-1, keepdims=True)
```

`dsink`는 커널 **밖에서 닫힌 형식**으로:

```python
sink_b = attn_sink[None,None,:,None]
p_sink = jnp.exp(sink_b - lse)                       # 가짜 key의 가중치
dsink  = (-p_sink * D).sum(axis=(0,1,3))             # [n_h]
```

유도: $o = N/Z$, $Z = \sum e^{\text{logit}} + e^{\text{sink}}$.
$\partial o/\partial\text{sink} = -o\cdot(e^{\text{sink}}/Z) = -o\cdot p_{\text{sink}}$. 따라서
$dL/d\text{sink}_h = \sum_{b,t} dout\cdot(-o\,p_{\text{sink}}) = -\sum_{b,t} p_{\text{sink}}\,D$.
sink는 분모, 즉 `lse`에만 영향을 주므로 커널 밖 JAX로 충분합니다.

## E.3 ★ 역전파 커널 `_flash_sink_kernel_bwd` (318~371행)

순전파와 **같은 격자**로 돌며, key 스텝마다:

```python
# 1) lse로 p 복원. 재계산
logits = _dot2(q, k, contract=(3,3)) * scale       # [1,BQ,n_h,BS]
p = jnp.exp(logits - lse)                          # p_s = exp(logit - lse)
p = jnp.where(mask_b, p, 0.0)

# 2) dp = Σ_c dout·k. V==K
dp = _dot2(dout, k, contract=(3,3))                # [1,BQ,n_h,BS]

# 3) softmax 역전파 항등식
dlogits = p * (dp - D)                             # [1,BQ,n_h,BS]

# 4) dq: 이 스텝 기여를 누적. 순전파 acc와 같은 패턴
dq_contrib = _dot2(dlogits, k, contract=(3,2)) * scale   # [1,BQ,n_h,c]
dq_scratch_ref[...] += dq_contrib

# 5) dk = QK기여 + PV기여. 둘 다 n_h를 내적으로 줄임
dk_qk = _dot2(dlogits, q, contract=(2,2)) * scale  # [1,BQ,BS,c]
dk_pv = _dot2(p, dout, contract=(2,2))             # [1,BQ,BS,c]
dk_ref[...] = (dk_qk + dk_pv).astype(dk_ref.dtype)
```

- **`dq`는 누적**: 한 query는 모든 key 스텝에 걸쳐 기여가 쌓이므로 scratch에 더하다가 마지막
  스텝에 `dq_ref`로 씁니다. 순전파 `acc`와 같은 패턴입니다.
- **`dk`는 타일별**: 각 `(b,qi,si)` 칸이 서로 다른 key 슬롯을 담당해 겹치지 않으므로 그냥 그
  자리에 씁니다. 두 칸이 같은 슬롯을 건드리지 않아 **결정적**이고 별도 동기화가 필요 없습니다.
- 점수 경로 `dk_qk`와 value 경로 `dk_pv` 둘을 더합니다. V==K라 key가 점수에도 쓰이고 value로도
  쓰이기 때문입니다.

```python
@pl.when(s == nsteps - 1)
def _finalize_dq():
    dq_ref[...] = dq_scratch_ref[...].astype(dq_ref.dtype)
```

## E.4 custom_vjp로 묶기 (`_make_flash_diffable`, 480~515행)

JAX에게 "이 Pallas 연산의 미분은 위 손수 짠 커널을 써라"라고 등록하는 부분.

```python
@jax.custom_vjp
def _flash(q, K_full, mask_f, attn_sink):
    mask_bool = mask_f > 0.5
    out, _lse = _flash_attn_with_sink_pallas(...)   # 순전파, lse는 버림
    return out

def _fwd(q, K_full, mask_f, attn_sink):
    mask_bool = mask_f > 0.5
    out, lse = _flash_attn_with_sink_pallas(...)
    return out, (q, K_full, mask_f, mask_bool, attn_sink, out, lse)  # residual 저장

def _bwd(res, dout):
    q, K_full, mask_f, mask_bool, attn_sink, out, lse = res
    dq, dk, dsink = _flash_attn_with_sink_pallas_bwd(...)
    return dq, dk, jnp.zeros_like(mask_f), dsink    # mask 자리는 0

_flash.defvjp(_fwd, _bwd)
```

- **왜 `pl.pallas_call`에 custom_vjp가 필수?** Pallas 연산은 TPU에서 기본 자동미분이 안 됩니다.
  그래서 직접 짠 bwd를 `defvjp`로 붙입니다.
- **mask를 fp32로 넘기는 트릭**: `custom_vjp`는 gradient dtype이 입력과 맞아야 하는데 bool은
  의미 있는 float gradient가 없습니다. 그래서 경계에서는 `mask_f`라는 fp32로 넘기고, 안에서
  `>0.5`로 bool을 복원하며, bwd에선 그 자리에 `zeros_like(mask_f)`를 둡니다.
- **gather는 이 경계 밖**의 순수 JAX라, `dK_full → dK_comp/dK_swa` 변환은 중복 선택 블록의
  scatter-add까지 포함해 자동미분이 알아서 처리합니다.

## E.5 [실습] gradient가 정말 맞는지 검증

`jax.grad`로 커널을 미분하고, **수치미분(finite difference)** 과 비교해봅니다 — gradient가
맞다는 가장 확실한 증거.

In [ ]:
import jax, jax.numpy as jnp
from dsv4.kernel import sparse_attn_kernel
# (B.3 실습의 q, K_comp, topk_idxs, K_swa, attn_sink 를 이어서)

def loss(q, sink):
    return jnp.sum(sparse_attn_kernel(q, K_comp, topk_idxs, K_swa, sink) ** 2)

# 해석적 gradient (커널의 custom_vjp 경유)
dq, dsink = jax.grad(loss, argnums=(0, 1))(q, attn_sink)
print("dq 유한?", bool(jnp.all(jnp.isfinite(dq))), " dsink:", dsink.shape)

# sink에 대한 수치미분과 비교 (h만큼 흔들어 차분)
h = 1e-3
g_num = jnp.array([
    (loss(q, attn_sink.at[i].add(h)) - loss(q, attn_sink.at[i].add(-h))) / (2*h)
    for i in range(n_h)
])
print("해석적 dsink:", dsink)
print("수치적   dsink:", g_num)
print("최대 차이:", float(jnp.max(jnp.abs(dsink - g_num))))
# dsink 크기가 ~45라 절대차 ~0.1대는 상대오차 ~0.4% — 중심차분(h=1e-3) 정밀도 한계 안.

**관찰**: 손수 짠 역전파 커널이 만든 `dsink`가 수치미분과 상대오차 ~0.4%로 일치하면,
§E.2~E.3의 수식이 옳다는 증거입니다. `jax.grad`가 `_bwd`를 호출하고, `dsink`는 §E.2의 닫힌
형식, `dq`는 §E.3의 커널에서 나옵니다.

> 🎯 **연습 E-A**: `loss`가 `K_comp`도 받게 수정하고 `argnums`에 `K_comp`를 넣어 `dK_comp`를
> 구해보세요. gather 경계 밖이라 자동미분이 처리하는 gradient입니다. 그래도 잘 나오는지 확인합니다.

---

# F. V2 조립 — 전체 CSA/HCA forward

여기부터 [`kernel_v2.py`](./kernel_v2.py). V2는 **새 attention 커널을 안 짭니다** — V1을 그대로
재사용하고, 그 **앞뒤 전처리까지 붙여 전체 forward**를 완성합니다.

## F.1 V2 설계 철학 한 줄

> **"평범한 GEMM·softmax는 JAX 컴파일러 XLA에 맡기고, 진짜 이득 나는 곳만 Pallas로."**

- 압축기·인덱서·RoPE·RMSNorm → 그냥 JAX. XLA가 이미 잘 컴파일합니다.
- attention 코어 → Pallas. V1을 재사용하며, 큰 중간 행렬 제거가 이득입니다.
- Sinkhorn 반복 → Pallas. §G에서 다루며, VMEM 상주가 이득입니다.

## F.2 CSA 전체 forward (`_csa_forward_v2`, 128~178행)

입력은 hidden state `H: [B,n,d]` 하나. `eager.csa_forward`와 구조가 동일하고 마지막 attention만
Pallas로 바뀝니다. 단계별 요약:

| 단계 | 함수 | 하는 일 | 수식 핵심 |
|------|------|---------|-----------|
| ① 압축 | `csa_compress` | m토큰 → 압축엔트리 1개, overlapped 2m | 2m 위치 softmax 가중평균 |
| ② 인덱서 | `lightning_indexer` + `topk_indices` | query별 중요 블록 top-k 선택 | $I[t,s]=\sum_h w_h\,\text{ReLU}(q_h\cdot k_s)$, causal |
| ③ 투영 | `H@W_DQ@W_UQ`, `H@W_swaK` | 저랭크 query, SWA 키 | 행렬곱 |
| ④ RoPE | `apply_partial_rope` | 마지막 64차원에 회전 위치인코딩 | 위치별 채널쌍 회전 |
| ⑤ SWA | `swa_gather` | 직전 n_win 토큰 키 모음 | 슬라이딩 윈도우 |
| ⑥ 정규화 | `rms_norm` | attention 직전 RMSNorm | $x\cdot\text{rsqrt}(\text{mean}(x^2)+\epsilon)$ |
| ⑦ attention | `sparse_attn_kernel_v2` | **V1 커널** | §D 그대로 |

**압축기** ①만 조금 더 봅시다. CSA는 블록 $i$가 자기 m토큰인 a 경로와 **직전 블록**의 m토큰인
b 경로를 같이 봐서 overlapped 2m을 만듭니다.

$$
w = \text{softmax}_{2m}([\,Z^a+B^a \,\|\, Z^b_{\text{prev}}+B^b\,]),\quad
C_{\text{comp}}[i] = \sum_j w^a_j C^a_j + \sum_j w^b_j C^b_{\text{prev},j}
$$

attention용 `K_comp`는 `c=512`, 인덱서용 `K_IComp`는 작은 `c_I=128`로 둘을 따로 만듭니다.

**인덱서 + top-k** ②가 sparse의 원천입니다. 전체 압축블록 중 query마다 중요한 `k`개만 골라
`topk_idxs`를 만들고, 부족하면 `-1`로 패딩합니다. 이것이 §D.3의 `_gather_concat_mask`로 들어갑니다.

> **요점**: `_csa_forward_v2`는 새 커널이 아니라 **조립 라인**입니다. 무거운 GEMM은 JAX에,
> 마지막 attention만 Pallas. **역전파도 따로 안 짭니다** — JAX 전처리는 자동미분, attention은
> V1 custom_vjp가 처리.

## F.3 HCA 전체 forward (`_hca_forward_v2`, 193~250행)

CSA와 거의 같고 **두 가지만 다릅니다**:

**(1) top-k 대신 "전부 선택"**: HCA는 압축률 `m_prime`이 커서 블록 수가 적습니다. 그래서 모든
블록을 dense하게 보는데, **별도 커널 없이 CSA와 똑같은 sparse 커널을 재사용**합니다 — "전부
선택" 인덱스를 만들어서:

```python
all_idx = jnp.broadcast_to(jnp.arange(n_blk), (B, n, n_blk))   # 0..n_blk-1 전부
causal  = jnp.arange(n_blk)[None,:] <= (jnp.arange(n)[:,None] // m_prime)  # 블록 causal
all_idx = jnp.where(causal, all_idx, -1)                       # 미래 블록은 -1
return sparse_attn_kernel_v2(q, K_comp, all_idx, K_swa, sink)  # 같은 커널!
```

레퍼런스는 dense MQA를 위해 `[B,n,n_h,n_blk]` 가중치를 통째로 만들어 메모리 병목이 되는데,
V2는 같은 결과를 FlashAttention으로 얻어 그 텐서를 만들지 않습니다.

**(2) `n_blk > 4096`이면 `NotImplementedError`**: 블록이 너무 많으면 현재 타일링으로 감당이
안 돼 streaming 커널은 V3+로 미룹니다. 레퍼런스도 같은 제한이 있습니다.

## F.4 [실습] 전체 CSA forward 돌려보기

In [ ]:
import jax, jax.numpy as jnp
from dsv4 import eager
from dsv4.kernel import csa_forward_kernel

cfg = eager.SMALL_CSA                 # 개발용 작은 프리셋 (d=128, c=128, n_h=4, ...)
B, n = 1, 16
H = jax.random.normal(jax.random.PRNGKey(1), (B, n, cfg.d))
params = eager.init_csa_params(jax.random.PRNGKey(2), cfg)

out_k = csa_forward_kernel(H, params, cfg)         # JAX 전처리 + Pallas attention
out_e = eager.csa_forward(H, params, cfg)          # 순수 JAX 레퍼런스
print("출력:", out_k.shape, " 차이:", float(jnp.max(jnp.abs(out_k - out_e))))
# 출력: (1, 16, 4, 128)  차이: 2.4e-07

**관찰**: `H` 하나가 들어가 압축→인덱서→top-k→RoPE→RMSNorm→attention을 거쳐
`[B,n,n_h,c]`가 나옵니다. 중간의 sparse 선택·FlashAttention을 다 거쳐도 레퍼런스와 `~2e-7`.
`DSV4_KERNEL=ref` 환경변수를 주면 `csa_forward_kernel`이 eager로 우회하니, 둘이 같은 표면임을
확인할 수 있습니다.

> 🎯 **연습 F-A**: `n=16`을 `n=12`로 바꿔보세요. `_v1_forward`의 패딩 `(-n)%bq`가 동작해
> 여전히 잘 도는지 확인합니다.

---

# G. V2 Sinkhorn — 새 Pallas 커널

드디어 V2가 **진짜 새로 짠** 커널입니다. 먼저 수학, 그다음 forward 커널, 마지막에 가장 영리한
역전파를 봅니다.

## G.1 이게 뭘 계산하나 — doubly stochastic 행렬

DeepSeek V4의 **mHC 잔차 매핑**은 논문 §2.2에 나오며, `hc`개의 병렬 잔차 스트림을 **서로
섞습니다**. 토큰당 3개를 만듭니다.

| 출력 | 모양 | 역할 | 범위 |
|------|------|------|------|
| `pre` | `[B,n,hc]` | 입력 게이트 | $\sigma \to (0,1)$ |
| `post` | `[B,n,hc]` | 출력 게이트 | $2\sigma \to (0,2)$ |
| `comb` | `[B,n,hc,hc]` | **스트림 혼합 행렬** | **doubly stochastic** |

`comb`가 주인공. 그냥 아무 행렬로 섞으면 잔차 크기가 폭주/소멸합니다. 그래서 **doubly
stochastic**으로 제약:

> **doubly stochastic**: 모든 원소 ≥0, **모든 행 합 = 1** 이면서 **모든 열 합 = 1** 인 정사각
> 행렬.
> 직관: "섞되 총량은 보존." 받는 가중치 합도 1, 주는 가중치 합도 1 → 에너지 보존 → 학습 안정.

## G.2 Sinkhorn–Knopp — 임의 행렬을 doubly stochastic으로

**정리**: 양수 행렬에 **행 정규화와 열 정규화를 번갈아 반복**하면 doubly stochastic으로 수렴.

- 행 정규화: 각 행을 행 합으로 나눔 → 행 합 1, 대신 열 합은 깨짐
- 열 정규화: 각 열을 열 합으로 나눔 → 열 합 1, 대신 행 합이 살짝 깨짐
- 번갈아 하면 그 "살짝"이 점점 줄어 둘 다 1로 수렴.

다음은 `_sinkhorn_body` 296~305행의 절차입니다.

```python
comb = comb_raw * hc_scale[2] + comb_bias       # affine 변환
comb = jax.nn.softmax(comb, axis=-1) + eps       # 1패스: 행 softmax. 양수화 + 행 합 1
comb = comb / (comb.sum(axis=-2, keepdims=True) + eps)   # 열 정규화
def step(_, c):                                  # 2패스~: 행→열 정규화 반복
    c = c / (c.sum(axis=-1, keepdims=True) + eps)        # 행 정규화. axis=-1이 행 합
    c = c / (c.sum(axis=-2, keepdims=True) + eps)        # 열 정규화. axis=-2가 열 합
    return c
comb = jax.lax.fori_loop(0, sinkhorn_iters - 1, step, comb)   # 18번 더
```

- `axis=-1` 합 = 행 합, `axis=-2` 합 = 열 합.
- 첫 패스만 softmax: 음수가 섞인 행렬을 한 번에 양수이면서 행 합이 1인 상태로 만듭니다. Sinkhorn은 양수를 전제하기 때문입니다.
- `eps`: 0-나눗셈 방지. `sinkhorn_iters=19`면 첫 패스 1 + 루프 18.

`pre`/`post`는 단순: $\text{pre}=\sigma(\text{pre\_raw}\cdot s_0+b)+\epsilon$,
$\text{post}=2\sigma(\cdots)$.

## G.3 왜 Pallas로? grid와 레이아웃

레퍼런스도 같은 걸 JAX로 합니다. 하지만 19번 반복하는 동안 `hc×hc` 행렬이 매번 **HBM을
왕복**합니다. Pallas로 내리면 그 행렬을 **VMEM에 눌러두고** 19번을 한 커널 안에서 돕니다.
§A.2 제1법칙입니다.

```python
grid = (B, n // bn)        # 토큰을 BN개씩. 누적 축 없음 — 토큰별 완전 독립 → 전부 "parallel"
```

§D의 attention과 달리 **누적 scratch도, 순차 축도 없습니다**. 한 프로그램이 자기 토큰들의
19-iter를 통째로 닫아버리기 때문.

**레이아웃 트릭** (`_mhc_sinkhorn_v2`, 567~575행): 커널 안에서 마지막 두 차원을 건드리는
`reshape`는 TPU 제약이 있어, `mixes`를 **커널 밖에서** 미리 셋으로 가릅니다:

```python
pre_raw  = mixes[..., :hc]                          # [B,n,hc]
post_raw = mixes[..., hc:2*hc]                        # [B,n,hc]
comb_raw = mixes[..., 2*hc:].reshape(B,n,hc,hc)       # [B,n,hc,hc]  ← 밖에서 reshape
```

## G.4 forward 커널 본체 (`_sinkhorn_body`, 269~309행)

```python
def _sinkhorn_body(pre_raw_ref, post_raw_ref, comb_raw_ref,
                   hc_scale_ref, pre_bias_ref, post_bias_ref, comb_bias_ref,
                   pre_ref, post_ref, comb_ref, *, sinkhorn_iters, eps):
    # 전부 fp32로 읽고
    pre  = jax.nn.sigmoid(pre_raw * hc_scale[0] + pre_bias) + eps
    post = 2.0 * jax.nn.sigmoid(post_raw * hc_scale[1] + post_bias)
    comb = comb_raw * hc_scale[2] + comb_bias
    comb = jax.nn.softmax(comb, axis=-1) + eps
    comb = comb / (comb.sum(axis=-2, keepdims=True) + eps)
    comb = jax.lax.fori_loop(0, sinkhorn_iters - 1, step, comb)
    pre_ref[...], post_ref[...], comb_ref[...] = pre, post, comb   # 써넣기
```

§G.2의 수식 그대로. grid의 각 `(b,ni)`에서 `BN`개 토큰의 `pre/post/comb`를 전부 계산.
`fori_loop`가 VMEM 안에서 반복.

## G.5 ★★ 역전파 — 3조각 분해 + Pallas 안의 jax.vjp

V2에서 제일 볼 만한 곳입니다. Sinkhorn forward는 19반복과 정규화 야코비안 때문에 미분이
까다롭지만, **세 조각으로 쪼개** 각각 가장 싸게 처리합니다. 코드는 `_mhc_sinkhorn_v2_bwd` 464~547행.

```
forward                            backward
────────────────────────────────  ─────────────────────────────────────────
pre  = σ(pre_raw·s0+b)+eps      →  (1) sigmoid bwd       [JAX, 닫힌 형식]
post = 2σ(post_raw·s1+b)        →  (1) sigmoid bwd       [JAX, 닫힌 형식]
comb_init = comb_raw·s2+b       →  (3) affine bwd        [JAX, 닫힌 형식]
comb = sinkhorn(comb_init, 19)  →  (2) 19-iter sinkhorn  [Pallas, jax.vjp]  ← 비싼 조각만
```

**조각 (1) sigmoid**, 닫힌 형식: $\sigma'(z)=\sigma(z)(1-\sigma(z))$ 한 줄.
```python
sig = jax.nn.sigmoid(z_pre)
dz  = dpre * sig * (1 - sig)
dpre_raw  = dz * hc_scale[0]       # chain: ∂z/∂pre_raw = s0
dpre_bias = dz.sum(axis=(0,1))     # bias는 전역 → 토큰·배치 합산
```

**조각 (2) 19-iter sinkhorn** — ★ 트릭, Pallas 안에서 `jax.vjp`:

먼저 **미분 가능한 순수 forward** `_sinkhorn_norm_pure`를 따로 정의합니다.
```python
def _sinkhorn_norm_pure(comb_init, *, sinkhorn_iters, eps):
    comb = jax.nn.softmax(comb_init, axis=-1) + eps
    comb = comb / (comb.sum(axis=-2, keepdims=True) + eps)
    def step(c, _):
        c = c / (c.sum(axis=-1, keepdims=True) + eps)
        c = c / (c.sum(axis=-2, keepdims=True) + eps)
        return c, None
    comb, _ = jax.lax.scan(step, comb, None, length=sinkhorn_iters - 1)  # ← scan!
    return comb
```

> **`fori_loop`(forward) vs `scan`(backward) — 왜 다른 루프?**
> `fori_loop`는 빠르지만 **미분이 안 됩니다**. `scan`은 역방향 자동미분을 네이티브로 지원하며
> 중간값을 residual로 자동 관리합니다. 같은 반복 횟수면 수치적으로 동일합니다. forward는 빠른
> 루프, backward 재계산은 미분되는 루프로 역할을 나눕니다.

그리고 그 순수 함수에 **Pallas 커널 안에서** `_sinkhorn_norm_body_bwd`로 `jax.vjp`를 겁니다.
```python
_, vjp_fn = jax.vjp(partial(_sinkhorn_norm_pure, ...), comb_init)
(dcomb_init,) = vjp_fn(dcomb)      # upstream dcomb → dcomb_init
```
`jax.vjp`가 forward를 한 번 돌리며 만든 19-iter 중간값이 전부 **VMEM scratch에 머뭅니다**. HBM에
19개 행렬을 다시 펼칠 필요가 없습니다. §A.2 제1법칙입니다. 손으로 야코비안 19번을 유도하는 것과
결과가 같으니 `jax.vjp`에 맡기는 게 정답입니다.

**조각 (3) affine**, 닫힌 형식: `comb_init = comb_raw·s2 + b`의 선형 역전파. 조각 (1)과 똑같은
chain rule.

마지막에 세 조각의 gradient를 forward가 쪼갠 순서대로 다시 합쳐 `dmixes`, `dhc_scale`, `dhc_base`를 만듭니다.

## G.6 심화: jax.vjp가 "내부에서" 하는 일 — 정규화 야코비안 손유도

조각 (2)에서 우리는 `jax.vjp`에 19-iter 미분을 떠넘겼습니다. 편하지만 블랙박스죠. 그 안에서
**실제로 무슨 미분이 일어나는지** 손으로 열어봅시다. 그러면 왜 forward 중간값을 저장해야
하는지가 눈으로 보입니다. forward 함수 `_sinkhorn_norm_pure`는 결국 정규화와 softmax 두 종류의
스텝을 연쇄한 것뿐이라, **그 둘의 VJP만 유도하면** 전체가 풀립니다.

> **VJP, 즉 벡터-야코비안 곱이란?** 역전파의 기본 단위. 한 연산 $y=f(x)$에 대해, 위에서 내려온
> cotangent $\bar y$를 받아 $\bar x = \big(\tfrac{\partial y}{\partial x}\big)^{\!\top}\bar y$를
> 돌려주는 것. 야코비안 행렬을 통째로 만들지 않고 곱만 계산하는 게 핵심.

**빌딩블록 1 — 정규화 한 스텝의 VJP**

정규화: $y_i = \dfrac{x_i}{s+\epsilon}$이고 $\;s=\sum_{k} x_k$는 정규화 축 위의 합입니다. 야코비안은
곱미분으로 구합니다.

$$
\frac{\partial y_i}{\partial x_j} = \frac{\delta_{ij}}{s+\epsilon} \;-\; \frac{x_i}{(s+\epsilon)^2}
$$

첫 항은 분자 $x_i$를 직접 미분한 것이고, 둘째 항은 분모 $s=\sum x$가 $x_j$를 품고 있어 생깁니다. VJP는:

$$
\bar x_j = \sum_i \bar y_i\frac{\partial y_i}{\partial x_j}
= \frac{\bar y_j}{s+\epsilon} - \frac{\sum_i \bar y_i x_i}{(s+\epsilon)^2}
$$

$x_i/(s+\epsilon)=y_i$ 를 대입하면 $\sum_i \bar y_i x_i/(s+\epsilon)=\sum_i \bar y_i y_i =: \sigma$.
정리:

$$
\boxed{\;\bar x = \frac{\bar y - \sigma}{\,s+\epsilon\,},\qquad \sigma=\sum_{\text{축}}\bar y\odot y\;}
$$

직관: **"받은 gradient $\bar y$ 에서, 출력분포 $y$로 가중평균한 양 $\sigma$를 빼고, 그 스텝의
분모로 나눈다."** $\sigma$는 정규화 축을 따라 합한 스칼라이며 그 축으로 브로드캐스트됩니다. 행 정규화면
축=−1, 열 정규화면 축=−2. 코드로:

```python
def norm_vjp(g, x, axis, eps):              # 검증된 형태. jax.vjp와 1e-16 일치
    s = x.sum(axis, keepdims=True) + eps
    y = x / s
    sigma = (g * y).sum(axis, keepdims=True)
    return (g - sigma) / s
```

**빌딩블록 2 — softmax의 VJP**

$y=\text{softmax}(x)$의 표준 결과이며 같은 방식으로 유도됩니다.

$$
\boxed{\;\bar x = y\odot(\bar y - \sigma),\qquad \sigma=\sum_{\text{축}}\bar y\odot y\;}
$$

빌딩블록 1과 형태가 거의 같고 앞에 $y\odot$ 만 붙습니다. forward의 `+eps`는 **덧셈 상수**라
gradient를 그대로 통과시킵니다. $\partial(c+\epsilon)/\partial c = 1$이기 때문입니다.

**연쇄 — 19-iter를 역순으로**

forward를 스텝 리스트로 펼치면:

$$
\underbrace{\text{softmax}}_{S}\;\to\;\underbrace{\text{col}}_{C_0}\;\to\;
\Big[\underbrace{\text{row}}_{R_t}\to\underbrace{\text{col}}_{C_t}\Big]_{t=1}^{18}
$$

backward는 출력 cotangent $d\text{comb}$에서 시작해 이 연쇄를 **거꾸로** 한 스텝씩 되돌립니다:

```
g ← dcomb
for t = 18 … 1:                          # 각 iteration을 역순으로
    g ← norm_vjp(g, x=(Cₜ의 입력), axis=-2)   # 열 정규화 되돌리기
    g ← norm_vjp(g, x=(Rₜ의 입력), axis=-1)   # 행 정규화 되돌리기
g ← norm_vjp(g, x=(C₀의 입력), axis=-2)       # pass0 열 정규화
g ← softmax_vjp(g, x=comb_init)               # softmax. +eps는 그냥 통과
dcomb_init ← g
```

여기서 각 `norm_vjp`/`softmax_vjp`는 **그 스텝의 forward 입력 $x$**, 따라서 그 스텝의 분모 $s$와
출력 $y$가 필요합니다. 이게 바로 **19개 중간값을 저장해야 하는 이유**입니다. 그리고 `jax.vjp`를
*Pallas 커널 안에서* 부르면 이 중간값들이 HBM이 아니라 **VMEM scratch에 머뭅니다**. §G.5 본문과
§A.2 제1법칙입니다. 즉 위 의사코드가 `jax.vjp(_sinkhorn_norm_pure)`가 내부에서 실제로 도는 모습이고,
손으로 짜든 `jax.vjp`에 맡기든 결과는 동일합니다.

## G.7 custom_vjp 구조 + 왜 kernel.py엔 안 두나

```python
@lru_cache(maxsize=None)
def _make_sinkhorn_kernel(config, hc, sinkhorn_iters, eps):
    @jax.custom_vjp
    def fn(mixes, hc_scale, hc_base): return _mhc_sinkhorn_v2(...)
    def _fwd(...): return out, (mixes, hc_scale, hc_base)   # residual = 원본 입력
    def _bwd(res, douts): return _mhc_sinkhorn_v2_bwd(...)
    fn.defvjp(_fwd, _bwd); return fn
```

`lru_cache`로 `(config, hc, iters, eps)`마다 캐싱하므로 이 값들이 트레이싱 상수가 됩니다.
residual은 원본 입력뿐입니다. backward가 순수 함수를 재실행하므로 중간값 저장이 필요 없습니다.

> **`kernel.py` 표면에 두 번째 custom_vjp를 안 두는 이유**: 그러면 `hc, sinkhorn_iters, eps`
> 같은 파이썬 기본값이 **tracer로 강제 변환**되어, 슬라이싱·`lru_cache` 키로 쓰는 순간
> 터집니다. 경계를 V1/V2 **안쪽**으로 좁게 두는 게 이 문제를 피하는 길.

## G.8 [실습] Sinkhorn 돌려보고 doubly stochastic 확인

In [ ]:
import jax, jax.numpy as jnp
from dsv4 import eager
from dsv4.kernel import mhc_sinkhorn_kernel

B, n, hc, iters, eps = 1, 16, 4, 19, 1e-6
mixes    = jax.random.normal(jax.random.PRNGKey(3), (B, n, (2+hc)*hc))
hc_scale = jnp.ones((3,))
hc_base  = jnp.zeros(((2+hc)*hc,))

pre_k, post_k, comb_k = mhc_sinkhorn_kernel(mixes, hc_scale, hc_base, hc, iters, eps)
pre_e, post_e, comb_e = eager.mhc_sinkhorn(mixes, hc_scale, hc_base, hc, iters, eps)

print("comb 모양:", comb_k.shape, " 레퍼런스 차이:", float(jnp.max(jnp.abs(comb_k - comb_e))))
# comb 모양: (1, 16, 4, 4)  레퍼런스 차이: 1.2e-07

# ★ doubly stochastic 확인: 한 토큰의 comb 행렬에서 행 합·열 합이 모두 ≈ 1
M = comb_k[0, 0]                       # [hc, hc] 한 토큰의 혼합 행렬
print("행 합:", jnp.round(M.sum(axis=-1), 4))   # ≈ [1, 1, 1, 1]
print("열 합:", jnp.round(M.sum(axis=-2), 4))   # ≈ [1, 1, 1, 1]

**관찰**: `comb`의 모든 행 합과 열 합이 `≈ 1.0` — Sinkhorn 반복이 실제로 doubly
stochastic을 만들어냈고, §G.2의 수렴이 눈으로 확인됩니다. 커널과 레퍼런스 차이는 `~1e-7`.

> 🎯 **연습 G-A**: `iters`를 19에서 `2`, `5`로 줄여보세요. 행 합·열 합이 1에서 얼마나 벗어나는지
> — 반복이 적으면 덜 수렴함을 직접 보세요.
> 🎯 **연습 G-B**: `jax.grad`로 미분해보세요:
> `jax.grad(lambda m: jnp.sum(mhc_sinkhorn_kernel(m, hc_scale, hc_base, hc, iters, eps)[2]**2))(mixes)`.
> §G.5의 3조각 역전파, 특히 Pallas 안 `jax.vjp`가 통째로 동작해 `dmixes`가 나오는지 확인합니다.
> 🎯 **연습 G-C, 검증**: §G.6의 `norm_vjp`/`softmax_vjp`를 그대로 구현해 역순 연쇄를 짜고,
> `jax.vjp(_sinkhorn_norm_pure)` 결과와 비교해보세요.
> `jax.config.update("jax_enable_x64", True)`로 돌리면 최대 차이가 기계 정밀도 수준인 **~1e-16**으로
> 떨어집니다. 손유도가 정확하다는 증거입니다. 이 저장소에서 실제 확인된 결과는 norm/softmax VJP 각
> `1e-16`, 전체 연쇄 `2.2e-16`입니다.

---

# 부록 A. Sinkhorn 수렴 직관

행 정규화 $R$, 열 정규화 $C$는 각각 행 합=1인 집합, 열 합=1인 집합이라는 두 볼록 집합으로의
**교대 사영**입니다. 양수성을 보존하고, 두 집합의 교집합인 doubly stochastic 집합이 비어있지
않으면 그 한 점으로 단조 수렴합니다. 그래서 19번이면 실용적으로 충분하며, §G.8 실습에서 행과 열
합이 ≈1임을 확인합니다. `eps`가 0-나눗셈을 막아 수치적으로 안전합니다. $\blacksquare$

online softmax의 정확성 증명은 §D.5 "왜 정확한가"에 본문으로 들어가 있습니다.

# 부록 B. 자주 헷갈리는 점 모음

| 질문 | 답 | 절 |
|------|------|------|
| value V는 어디 있나? | 없음. V==K라 PV 곱에서 `k`를 그대로 씀 | C.2 |
| `m,l`을 왜 `(n_h,c)`로 부풀리나? | `n_h`가 128 배수 레인 정렬을 위반 → `c` 레인에 복사 | D.4 |
| mask를 왜 `-inf`가 아닌 `-1e30`으로? | 전체 mask 시 `exp(-inf - -inf)=nan` 방지 | D.6 |
| `lse`는 순전파에 안 쓰는데 왜 저장? | 역전파에서 `p=exp(logit-lse)` 복원용 | D.6, E.1 |
| gather는 왜 커널 밖 JAX? | 자동미분이 `dK_comp/dK_swa`를 공짜로 처리 | D.3, E.4 |
| 블록 크기를 바꾸면 답이 변하나? | 안 변함. 속도/메모리 튜닝일 뿐 | D.5, D.8 |
| `fori_loop` vs `scan`? | forward=빠름·미분불가 / backward=미분가능 | G.5 |
| Pallas op엔 왜 custom_vjp 필수? | Pallas는 기본 자동미분 안 됨 | E.4, G.7 |

# 부록 C. 더 읽을거리 / 다음 단계

- 코드 헤더 주석들: [`kernel_v1.py`](./kernel_v1.py) 1~100행, [`kernel_v2.py`](./kernel_v2.py)
  1~82행에 설계 근거와 V3+ TODO가 정리돼 있습니다. 스칼라-프리페치 gather, streaming HCA, 메가코어
  파이프라이닝, 오토튜닝 등입니다.
- 레퍼런스 [`eager.py`](./eager.py): 모든 커널의 정답 기준. 헷갈리면 여기 순수 JAX 버전과 비교하세요.
- 설정 [`kernel_config.py`](./kernel_config.py): TPU 세대별 블록 크기 자동 결정. `config_for`가
  VMEM 예산 안에서 최적 `(BQ, BS)`를 고르는 방식.
- 실행 팁: `DSV4_KERNEL=ref` 환경변수로 모든 진입점을 eager로 우회할 수 있습니다. 디버그용입니다.